# Practice 4-5. 밀집 검색 (Dense Retrieval) — pgvector 벡터 인덱싱

책은 `OpenAIEmbeddings(text-embedding-3-large)`로 임베딩한 뒤 `FAISS.from_documents()`로 인메모리 인덱스를 만들고
`save_local`/`load_local`로 파일에 저장합니다. FAISS 인덱스는 결국 한 프로세스의 메모리 위에 얹히는 구조라서, 서버를
재시작하거나 다른 프로세스에서 검색하려면 파일 전체를 다시 읽어 들여야 하고, 억 단위로 커지면 애초에 한 장비의 메모리에
올라가지 않습니다.

이 노트북은 로컬 LLM 서버(LM Studio)의 임베딩 모델로 벡터를 만들고, **pgvector에 진짜 ANN(근사 최근접 이웃) 인덱스인
HNSW를 구축**해서 밀집 검색을 구현합니다. 인덱스와 벡터는 PostgreSQL 서버(컨테이너 `pgvector`)에 영구적으로 저장되므로,
어떤 프로세스에서 접속하든 동일한 인덱스를 그대로 씁니다.

## 0. 환경 설정

In [ ]:
import os
import re
import psycopg
from urllib.parse import quote_plus

# --- LM Studio ---
LMSTUDIO_BASE_URL = os.getenv("LMSTUDIO_BASE_URL", "http://...:.../v1")
LMSTUDIO_API_KEY = os.getenv("LMSTUDIO_API_KEY", "lm-studio")

EMBED_MODEL = "text-embedding-qwen3-embedding-8b"
LLM_MODEL   = "qwen3-30b-a3b-instruct-2507"   # LM Studio에 로드되어 있는 로컬 채팅 모델

# --- PostgreSQL (pgvector 이미지, Docker 컨테이너) ---
PG_HOST = os.getenv("PG_HOST", "pgvector")
PG_PORT = int(os.getenv("PG_PORT", "5432"))
PG_DB = os.getenv("PG_DB", "admin")
PG_USER = os.getenv("PG_USER", "admin")
PG_PASSWORD = os.getenv("PG_PASSWORD", "...")
PG_DSN = f"host={PG_HOST} port={PG_PORT} dbname={PG_DB} user={PG_USER} password={PG_PASSWORD}"
# langchain_postgres 는 SQLAlchemy 비동기 엔진(asyncpg)을 쓰므로 URL 형식으로도 준비해 둔다.
PG_ASYNC_URL = f"postgresql+asyncpg://{PG_USER}:{quote_plus(PG_PASSWORD)}@{PG_HOST}:{PG_PORT}/{PG_DB}"

# --- 입력 파일 (노트북 기준 상대경로) ---
DATA_PATH = "투자설명서.pdf"


def strip_think(text: str) -> str:
    """qwen3 등 추론 모델의 <think>...</think> 블록 제거."""
    if text is None:
        return ""
    return re.sub(r"(^|<think>).*?</think>", "", text, flags=re.DOTALL).strip()


In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

llm = ChatOpenAI(
    model=LLM_MODEL,
    base_url=LMSTUDIO_BASE_URL,
    api_key=LMSTUDIO_API_KEY,
    temperature=0,
    # qwen3 계열 추론 모델은 답변 전에 <think> 추론 토큰을 많이 소모하므로 넉넉히 잡는다
    # (1024로는 추론만 하다가 잘려 답변이 빈 문자열로 나오는 경우가 있었다)
    max_tokens=2048,
)

# check_embedding_ctx_length=False : tiktoken 대신 원문 문자열을 전송 → LM Studio 호환
# chunk_size=8 : 로컬 LM Studio 서버가 한 번에 받는 텍스트 수가 많으면 연결이 끊기므로 작게 배치
base_embeddings = OpenAIEmbeddings(
    model=EMBED_MODEL,
    base_url=LMSTUDIO_BASE_URL,
    api_key=LMSTUDIO_API_KEY,
    check_embedding_ctx_length=False,
    chunk_size=8,
)

with psycopg.connect(PG_DSN) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT version();")
        print("PostgreSQL:", cur.fetchone()[0].split(",")[0])

EMBED_DIM = len(base_embeddings.embed_query("연결 테스트"))
print("EMBED:", EMBED_DIM, "차원")
print("LLM  :", strip_think(llm.invoke("연결 테스트. 'ok' 라고만 답하세요.").content)[:50])


## 1. 문서 로드 · 분할

책과 동일하게 `투자설명서.pdf`를 `PyPDFLoader`로 읽어 `RecursiveCharacterTextSplitter`(chunk_size=2000, chunk_overlap=200)로 분할합니다.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader(DATA_PATH)
doc_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)

docs = loader.load_and_split(doc_splitter)
print("분할된 문서 수:", len(docs))


## 2. pgvector 인덱스의 차원 제한

pgvector의 근사 최근접 이웃 인덱스(HNSW, IVFFlat)는 컬럼 차원에 상한이 있습니다(`vector` 타입 기준 HNSW/IVFFlat 모두 2000차원,
절반 정밀도 `halfvec` 타입을 쓰면 4000차원까지). 반면 로컬 임베딩 모델(`embedding-8b:sl`)은 4096차원을 그대로 내놓습니다.
먼저 4096차원 그대로 HNSW 인덱스를 만들어 실제로 어떤 오류가 나는지 확인합니다.

In [ ]:
try:
    with psycopg.connect(PG_DSN, autocommit=True) as conn:
        with conn.cursor() as cur:
            cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
            cur.execute(f"CREATE TABLE IF NOT EXISTS _dim_probe (id serial primary key, embedding vector({EMBED_DIM}));")
            cur.execute("CREATE INDEX IF NOT EXISTS _dim_probe_hnsw ON _dim_probe USING hnsw (embedding vector_cosine_ops);")
    print("인덱스 생성 성공 (예상 밖)")
except Exception as e:
    print("인덱스 생성 실패 (예상된 결과):", type(e).__name__, "-", str(e).strip())
finally:
    with psycopg.connect(PG_DSN, autocommit=True) as conn:
        with conn.cursor() as cur:
            cur.execute("DROP TABLE IF EXISTS _dim_probe;")


실무에서 이 제약을 다루는 방법은 크게 두 가지입니다.

1. 애초에 인덱싱 가능한 차원(≤2000, `halfvec`이면 ≤4000)의 임베딩 모델을 고른다.
2. 이미 정해진 임베딩 모델의 출력을 **차원 축소**해서 저장·인덱싱한다.

이 노트북의 임베딩 모델(`embedding-8b:sl`, Qwen3 임베딩 계열)은 **MRL(Matryoshka Representation Learning)**로 학습되어
있습니다. MRL로 학습된 임베딩은 벡터의 앞쪽 차원만 잘라내도(뒤쪽을 버려도) 그 자체로 유효한 저차원 임베딩이 되도록
설계되어 있어서, 별도의 학습된 투영 행렬이나 근사 기법 없이 **그냥 앞에서부터 잘라내면(truncation)** 됩니다. 코사인
유사도는 벡터의 방향만 비교하므로 길이가 줄어도 재정규화가 필요 없고(pgvector의 코사인 연산자가 내부적으로 정규화합니다),
행렬을 학습·저장할 필요도 없어 랜덤 프로젝션보다 훨씬 간단합니다. (MRL을 지원하지 않는 일반 임베딩 모델이라면 이렇게
단순히 잘라내는 방식은 유사도를 크게 훼손하므로 랜덤 프로젝션 같은 기법이 필요합니다.)

In [ ]:
from langchain_core.embeddings import Embeddings

PROJ_DIM = 1024  # HNSW(vector) 상한 2000보다 충분히 작게


class TruncatedEmbeddings(Embeddings):
    """MRL(Matryoshka Representation Learning)로 학습된 임베딩의 앞쪽 out_dim개 차원만 사용해
    pgvector HNSW 인덱스 상한 안으로 맞추는 래퍼. 코사인 거리는 방향만 비교하므로 재정규화가 필요 없다."""

    def __init__(self, base: Embeddings, out_dim: int):
        self.base = base
        self.out_dim = out_dim

    def embed_documents(self, texts: list) -> list:
        return [v[: self.out_dim] for v in self.base.embed_documents(texts)]

    def embed_query(self, text: str) -> list:
        return self.base.embed_query(text)[: self.out_dim]


embeddings = TruncatedEmbeddings(base_embeddings, out_dim=PROJ_DIM)
print(f"{EMBED_DIM}차원 → 앞쪽 {PROJ_DIM}차원만 사용 (MRL truncation) 준비 완료")


## 3. pgvector 벡터 테이블 · HNSW 인덱스 구축

`langchain_postgres`의 `PGEngine`/`PGVectorStore`(v2 API)로 벡터 테이블을 만들고 데이터를 적재한 뒤, `apply_vector_index()`로
실제 HNSW 인덱스를 생성합니다. `overwrite_existing=True`로 테이블을 매번 새로 만들기 때문에 몇 번을 다시 실행해도 안전합니다.

In [ ]:
!pip install langchain_postgres

In [ ]:
from langchain_postgres import PGEngine, PGVectorStore
from langchain_postgres.v2.indexes import HNSWIndex, DistanceStrategy

DENSE_TABLE = "practice4_dense_docs"

engine = PGEngine.from_connection_string(url=PG_ASYNC_URL)

# 재실행 안전: 테이블을 지우고 새로 만든다
engine.init_vectorstore_table(
    table_name=DENSE_TABLE,
    vector_size=PROJ_DIM,
    overwrite_existing=True,
)
print("벡터 테이블 준비 완료:", DENSE_TABLE)

vectorstore = PGVectorStore.create_sync(
    engine=engine,
    embedding_service=embeddings,
    table_name=DENSE_TABLE,
    distance_strategy=DistanceStrategy.COSINE_DISTANCE,
    k=2,
)
print("벡터스토어 준비 완료")


In [ ]:
vectorstore.add_documents(docs)
print("문서 적재 완료:", len(docs), "건")


In [ ]:
hnsw_index = HNSWIndex(
    name="practice4_dense_docs_hnsw",
    distance_strategy=DistanceStrategy.COSINE_DISTANCE,
    m=16,
    ef_construction=64,
)
vectorstore.apply_vector_index(hnsw_index)
print("HNSW 인덱스 유효성:", vectorstore.is_valid_index("practice4_dense_docs_hnsw"))


### 3.1 재실행 안전성 직접 증명 — 테이블을 한 번 더 초기화·재구축

`overwrite_existing=True`는 매 실행마다 `DROP TABLE IF EXISTS` 후 다시 `CREATE TABLE`을 실행합니다. 말로만 그렇다고 하지
않고, 이 노트북 안에서 테이블 초기화·적재·인덱싱을 한 번 더 반복해서 행 수가 중복되지 않고 그대로인지 직접 확인합니다.

In [ ]:
def rebuild_dense_table_and_count():
    engine.init_vectorstore_table(
        table_name=DENSE_TABLE,
        vector_size=PROJ_DIM,
        overwrite_existing=True,  # 기존 테이블을 DROP TABLE IF EXISTS 후 재생성
    )
    vs = PGVectorStore.create_sync(
        engine=engine,
        embedding_service=embeddings,
        table_name=DENSE_TABLE,
        distance_strategy=DistanceStrategy.COSINE_DISTANCE,
        k=2,
    )
    vs.add_documents(docs)
    vs.apply_vector_index(
        HNSWIndex(name="practice4_dense_docs_hnsw", distance_strategy=DistanceStrategy.COSINE_DISTANCE, m=16, ef_construction=64)
    )
    with psycopg.connect(PG_DSN) as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT count(*) FROM {DENSE_TABLE};")
            row_count = cur.fetchone()[0]
    return vs, row_count


vectorstore, row_count_2nd = rebuild_dense_table_and_count()

print("1차 적재 후 문서 수:", len(docs))
print("2차(재구축) 적재 후 문서 수:", row_count_2nd)

assert row_count_2nd == len(docs), "재구축 후 행 수가 분할 문서 수와 다릅니다 — 테이블이 중복 적재되었을 수 있습니다"
print("\n재구축 결과가 원본 문서 수와 정확히 일치합니다 — 테이블 초기화·재적재·재인덱싱을 몇 번 반복해도 안전합니다.")


## 4. 검색 확인

앞선 희소 검색(`04-postgresql-bm25-sparse-retrieval.ipynb`)과 같은 질문으로 유사도 검색을 해 봅니다.

In [ ]:
QUESTION = "이 회사가 발행한 주식의 총 발행량이 어느 정도야?"

results = vectorstore.similarity_search_with_score(QUESTION, k=5)
print(f"검색 결과 {len(results)}건")
for doc, dist in results:
    print(f"  [page={doc.metadata.get('page')} distance={dist:.4f}] {doc.page_content[:60]}")


## 5. LangChain 리트리버로 래핑

`PGVectorStore`는 `VectorStore`를 상속하므로 `as_retriever()`가 바로 됩니다. 책의 `faiss_retriever = vectordb.as_retriever(search_kwargs={"k": 2})`에 대응합니다.

In [ ]:
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

found = dense_retriever.invoke(QUESTION)
print(f"검색 결과 {len(found)}건")
for d in found:
    print(" -", d.metadata, d.page_content[:50])


## 6. 답변 생성 (RetrievalQA)

In [ ]:
try:
    from langchain_classic.chains import RetrievalQA
except ImportError:
    from langchain.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=dense_retriever,
    return_source_documents=True,
)

result = qa_chain.invoke({"query": QUESTION})

print("질문:", QUESTION)
print("답변:", strip_think(result["result"]))
print("\n사용된 문서:")
for d in result["source_documents"]:
    print(" -", d.metadata)


## 7. 관찰 — 의미 기반 검색이 표현 차이를 넘어선다

`04-postgresql-bm25-sparse-retrieval.ipynb`의 순수 키워드 기반 BM25(`k=2`)는 질의 "총 발행량"과 표현이 다른 정답 문단(489쪽, "발행주식총수(A) 보통주
13,602,977")을 놓쳤습니다. 이 노트북의 밀집 검색은 같은 질문에 대해 1위로 328쪽 문단("Ⅳ. 발행주식의 총수 ... 13,602,977")을
찾아냈고, 그 결과 `RetrievalQA`도 "13,602,977주"라는 정확한 숫자를 답했습니다. 질의에 "발행량"이라는 표현이 그대로 없어도
"발행주식의 총수"처럼 의미가 가까운 문단을 임베딩 유사도로 찾아낸 것입니다.

다만 밀집 검색은 표현이 완전히 다른 숫자·고유명사·법조문 번호 같은 정확 매칭에는 오히려 약할 수 있습니다. 다음
노트북(`06-bm25-pgvector-ensemble-retrieval.ipynb`)에서는 이 둘을 앙상블로 함께 써서 서로의 약점을 보완합니다.